# PCB Defect Synthesis & Detection Pipeline (Kaggle Edition - GPU 2x T4)
**Paper**: *"A Synthetic Defect Image Generation Method for Enhanced Industrial Defect Detection Based on Inpainting Diffusion and Context-aware Mask Generation"* (Journal of Intelligent Manufacturing, 2026)

---
### 🚀 Ưu Điểm Khi Chạy Trên Kaggle So Với Google Colab:
1. **Chế Độ Chạy Ngầm (Background Execution - Save & Run All)**: 
   - Google Colab yêu cầu phải giữ màn hình máy tính luôn mở (`on-screen`), nếu tắt máy hoặc ngắt mạng thì session sẽ bị ngắt kết nối.
   - **Kaggle cho phép tắt máy hoàn toàn**: Bạn chỉ cần bấm **`Save Version`** -> Chọn **`Save & Run All (Commit)`**. Kaggle sẽ tự động chạy toàn bộ pipeline trên máy chủ đám mây trong tối đa 9-12 giờ. Bạn có thể tắt máy đi ngủ hoặc làm việc khác!
2. **Cấu Hình Phần Cứng Mạnh Mẽ**: Miễn phí **GPU 2x T4 (Dual GPU - 30GB VRAM)** hoặc P100 (16GB VRAM) với hạn mức 30 giờ/tuần.
3. **Tích Hợp TQDM & Checkpoint Resume**: Tự động hiển thị thanh tiến trình trực quan (%) và hỗ trợ cờ `--resume` giúp lưu trạng thái, tự động bỏ qua các bước đã hoàn tất nếu cần chạy lại.

---
### ⚙️ Thiết Lập Môi Trường Kaggle Ban Đầu (BẮT BUỘC):
Trước khi chạy, hãy kiểm tra menu bên phải (**Notebook Settings**):
- **Accelerator**: Chọn **GPU T4 x2** (hoặc GPU P100)
- **Internet**: Bật **On** (để clone mã nguồn và tải model HuggingFace)
- **Persistence**: Chọn **Files only** (khuyên dùng)

## Bước 1: Kiểm Tra Cấu Hình Phần Cứng GPU (Dual GPU T4 x2)

In [ ]:
# ── 1. Kiểm tra cấu hình GPU ─────────────────────────────
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"GPU Device Count: {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} - {props.total_memory / (1024**3):.1f} GB VRAM")

if device_count >= 2:
    print("\n=> Cấu hình GPU 2x T4 đã sẵn sàng! Pipeline sẽ tận dụng tối đa tài nguyên.")
elif device_count == 1:
    print("\n=> Phát hiện 1 GPU. Bạn có thể chuyển sang GPU T4 x2 trong Settings -> Accelerator.")
else:
    print("\n=> CẢNH BÁO: Chưa bật GPU! Hãy vào Settings -> Accelerator -> GPU T4 x2.")

## Bước 2: Chuẩn Bị Mã Nguồn & Cài Đặt Thư Viện Phụ Thuộc

In [ ]:
# ── 2. Clone mã nguồn dự án vào thư mục /kaggle/working ──
import os
from pathlib import Path

working_dir = Path("/kaggle/working")
repo_dir = working_dir / "diffusion"

if not (working_dir / "run_pipeline.py").exists() and not (repo_dir / "run_pipeline.py").exists():
    print("Đang clone mã nguồn từ GitHub...")
    !git clone https://github.com/amebett1/diffusion.git /kaggle/working/diffusion

if (repo_dir / "run_pipeline.py").exists():
    os.chdir(repo_dir)
    print(f"Đang làm việc tại: {os.getcwd()}")
    !git pull
else:
    print(f"Đang làm việc tại: {os.getcwd()}")

# Cài đặt các thư viện cần thiết
%pip install -q diffusers transformers accelerate peft ultralytics opencv-python pyyaml gdown tqdm

## Bước 3: Chuẩn Bị Dataset DeepPCB
Notebook hỗ trợ 2 cách đưa dữ liệu vào:
- **Cách 1 (Kaggle Input)**: Nếu bạn đã upload dataset lên Kaggle Datasets hoặc gắn dataset có sẵn từ `/kaggle/input`.
- **Cách 2 (Google Drive)**: Tự động tải file zip từ Google Drive thông qua `gdown` và giải nén.

In [ ]:
# ── 3. Tải hoặc liên kết dữ liệu PCB ────────────────────────
import os
import shutil
import zipfile
from pathlib import Path

raw_dir = Path("data/pcb-defect-dataset")
raw_dir.mkdir(parents=True, exist_ok=True)

# TÙY CHỌN: Nếu bạn đã gắn dataset trên Kaggle (+ Add Input), bạn có thể:
# - Để trống ("") để notebook tự động phát hiện dataset trong /kaggle/input
# - Hoặc điền đường dẫn cụ thể, ví dụ: "/kaggle/input/pcb-defect-dataset"
CUSTOM_KAGGLE_INPUT = ""

kaggle_input = Path("/kaggle/input")
found_in_kaggle = False

# 1. Kiểm tra dataset được gắn trực tiếp trên Kaggle (+ Add Input)
if CUSTOM_KAGGLE_INPUT and Path(CUSTOM_KAGGLE_INPUT).exists():
    src_path = Path(CUSTOM_KAGGLE_INPUT)
    print(f"Đang liên kết dataset chỉ định tại: {src_path}")
    for item in src_path.iterdir():
        dst = raw_dir / item.name
        if not dst.exists():
            try:
                os.symlink(str(item.resolve()), str(dst.resolve()))
            except Exception:
                if item.is_dir():
                    shutil.copytree(str(item), str(dst), dirs_exist_ok=True)
                else:
                    shutil.copy2(str(item), str(dst))
    found_in_kaggle = True
    print(f"Đã liên kết dataset từ Kaggle Input vào '{raw_dir}' thành công!")
elif kaggle_input.exists():
    # Quét nếu có file zip trong /kaggle/input
    for zip_p in kaggle_input.glob("**/*.zip"):
        print(f"Phát hiện file ZIP trong Kaggle Input: {zip_p}")
        with zipfile.ZipFile(zip_p, "r") as z:
            names = z.namelist()
            has_root = any(n.startswith("pcb-defect-dataset/") for n in names)
            dest = Path("data") if has_root else raw_dir
            z.extractall(dest)
        found_in_kaggle = True
        print(f"Đã giải nén {zip_p.name} vào {dest}!")
        break

    # Quét nếu có thư mục ảnh trực tiếp trong /kaggle/input
    if not found_in_kaggle and any(kaggle_input.glob("**/*.jpg")):
        sample_jpg = next(kaggle_input.glob("**/*.jpg"))
        candidate = sample_jpg.parent
        while candidate != kaggle_input and candidate.parent != kaggle_input:
            if any((candidate / s).exists() for s in ["train", "val", "test", "images"]):
                break
            candidate = candidate.parent
        dataset_source_dir = candidate
        print(f"Tự động phát hiện dataset trong Kaggle Input: {dataset_source_dir}")
        for item in dataset_source_dir.iterdir():
            dst = raw_dir / item.name
            if not dst.exists():
                try:
                    os.symlink(str(item.resolve()), str(dst.resolve()))
                except Exception:
                    if item.is_dir():
                        shutil.copytree(str(item), str(dst), dirs_exist_ok=True)
                    else:
                        shutil.copy2(str(item), str(dst))
        found_in_kaggle = True
        print(f"Đã liên kết dataset từ Kaggle Input vào '{raw_dir}' thành công!")

# 2. Nếu không có sẵn trên Kaggle thì tải từ Google Drive qua gdown
DATASET_DRIVE_LINK = "https://drive.google.com/file/d/1jpg-Igk_2i8oM6B7N2_mPv6tM7UQJ34y/view?usp=sharing"

if not any(raw_dir.glob("**/*.jpg")):
    print("Không tìm thấy trong Kaggle Input. Đang tải dataset từ Google Drive (khoảng 1.1 GB)...")
    !gdown --fuzzy "{DATASET_DRIVE_LINK}" -O dataset.zip
    print("Đang giải nén dataset vào data/pcb-defect-dataset/...")
    with zipfile.ZipFile("dataset.zip", "r") as z:
        names = z.namelist()
        has_root = any(n.startswith("pcb-defect-dataset/") for n in names)
        dest = Path("data") if has_root else raw_dir
        z.extractall(dest)
    for s in ["train", "val", "test", "images"]:
        if (Path("data") / s).exists():
            shutil.move(str(Path("data") / s), str(raw_dir / s))
    print("Đã tải và giải nén dataset thành công!")

total_imgs = len(list(raw_dir.glob("**/*.jpg")))
print(f"Dữ liệu PCB đã sẵn sàng! Tìm thấy {total_imgs} ảnh trong '{raw_dir}'.")


## Bước 4: Tiền Xử Lý Dữ Liệu & Tạo 799 Ảnh Phôi Sạch (Telea Inpainting)
Quy trình tự động thực hiện:
1. Phân chia tập dữ liệu **In-Distribution (ID 10% Low-Data)** và **New Product Introduction (NPI)** theo thiết lập thực nghiệm của bài báo.
2. Áp dụng thuật toán Telea Inpainting để xóa sạch khuyết tật thực tế, sinh chính xác **799 ảnh phôi sạch không lỗi** (`data/pcb_normal/`) theo đúng trang 6 cột 1 của bài báo.
3. Xuất file cấu hình dataset `data/id_setup.yaml` và `data/train_metadata.json`.

In [ ]:
# ── 4. Tiền xử lý dữ liệu và tạo 799 ảnh normal ──────────────
!python run_pipeline.py prepare_data --raw_dir data/pcb-defect-dataset --output_dir data

# Kiểm tra bảng trạng thái pipeline
!python run_pipeline.py status

---
# 🌟 PHƯƠNG ÁN A: CHẠY TOÀN BỘ TRONG NỀN (RECOMMENDED)
### Dành cho Kaggle: **Save Version -> Save & Run All (Commit)**

> **Hướng Dẫn Tắt Máy:**
> 1. Bạn chỉ cần ấn nút **`Save Version`** ở góc trên cùng bên phải.
> 2. Trong hộp thoại hiện ra, chọn **`Save & Run All (Commit)`**.
> 3. Bấm **`Save`**.
> 4. **BÂY GIỜ BẠN CÓ THỂ ĐÓNG TRÌNH DUYỆT VÀ TẮT MÁY TÍNH!**
>
> Máy chủ Kaggle sẽ chạy ngầm lệnh bên dưới trong khoảng 7-9 giờ. Nhờ cờ `--resume`, nếu có bất kỳ gián đoạn nào, khi chạy lại pipeline sẽ tự động tiếp tục công đoạn chưa xong mà không phải train lại từ đầu.

In [ ]:
# ── ONE-CLICK FULL PIPELINE RUN (BACKGROUND / COMMIT) ─────────
!python run_pipeline.py full \
    --image_dir data/pcb_normal \
    --real_defect_dir data \
    --output_dir outputs/pcb_synthesis \
    --num_samples 829 \
    --run_benchmark \
    --resume

---
# 🔬 PHƯƠNG ÁN B: CHẠY TỪNG BƯỚC THỦ CÔNG (MODULAR EXECUTION)
Nếu bạn muốn thử nghiệm tương tác từng công đoạn và quan sát kết quả chi tiết, hãy chạy lần lượt các ô bên dưới.

### Step 1: Context-Aware Mask Generation (CAMG)
Sử dụng DINOv2 backbone để trích xuất đặc trưng vị trí ngữ nghĩa và thuật toán Top-k Sampling ($k=5$) để sinh mask khuyết tật hợp lý.

In [ ]:
# Huấn luyện CAMG prediction heads (10 epochs, batch 32, lr 1e-4)
!python run_pipeline.py step1_train --metadata_path data/train_metadata.json --epochs 10 --batch_size 32

# Sinh mask khuyết tật cho 799 ảnh
!python run_pipeline.py step1 --image_dir data/pcb_normal --output_dir outputs/pcb_synthesis --k 5

### Step 2a: Fine-tuning LoRA & Textual Inversion
Huấn luyện LoRA ($r=8, \alpha=16$) trên U-Net và Text Encoder cùng Multi-token Textual Inversion (6 placeholder tokens đại diện cho 6 loại lỗi PCB).

In [ ]:
!python run_pipeline.py step2_train \
    --metadata_path data/train_metadata.json \
    --output_dir outputs/pcb_synthesis \
    --lora_epochs 10 \
    --lora_batch 32 \
    --lr_unet 1e-4 \
    --lr_text_encoder 5e-5 \
    --lr_ti 5e-4

### Step 2b: Diffusion Inpainting Synthesis
Sinh tổng cộng 829 ảnh lỗi tổng hợp bằng Stable Diffusion Inpainting với 50 DDIM denoising steps (~4 giây/ảnh trên GPU T4).

In [ ]:
!python run_pipeline.py step2_generate \
    --mask_meta outputs/pcb_synthesis/masks/mask_metadata.json \
    --output_dir outputs/pcb_synthesis \
    --num_samples 829 \
    --lora_ckpt outputs/pcb_synthesis/lora_checkpoints/epoch_010 \
    --inpaint_steps 50

### Step 3: Confidence-Based Quality Filtering (YOLO11s)
Huấn luyện mô hình thẩm định chất lượng YOLO11s (100 epochs, AdamW) trên tập train 10% Low-Data. Sau đó tính điểm tin cậy trung bình trên từng vùng lỗi và loại bỏ 25% ảnh có chất lượng thấp nhất.

In [ ]:
!python run_pipeline.py step3 \
    --gen_meta outputs/pcb_synthesis/generated/generated_metadata.json \
    --output_dir outputs/pcb_synthesis \
    --data_yaml data/id_setup.yaml \
    --yolo_size s \
    --yolo_epochs 100 \
    --reject_ratio 0.25

### Step 4: Downstream Benchmark (Đánh Giá Mô Hình Hạ Nguồn YOLO11)
Huấn luyện và so sánh mô hình phát hiện lỗi trên:
- **Baseline Dataset**: Chỉ dùng 10% ảnh lỗi thực tế gốc.
- **Augmented Dataset**: Kết hợp 10% ảnh thực tế + ảnh tổng hợp đã qua lọc chất lượng từ Step 3.
- Tham số theo chuẩn bài báo: AdamW, `lr0 = 1e-4`, `epochs = 100`, `batch_size = 8`, ngưỡng đánh giá `conf = 0.25, iou = 0.25`.

In [ ]:
!python run_pipeline.py step4 \
    --models yolo11 \
    --base_yaml data/id_setup.yaml \
    --aug_yaml outputs/pcb_synthesis/filtered/augmented_dataset.yaml \
    --output_dir outputs/pcb_synthesis \
    --epochs 100 \
    --batch_size 8 \
    --lr0 1e-4 \
    --conf 0.25 \
    --iou 0.25

---
## Bước 5: Xem Kết Quả & Trực Quan Hóa Báo Cáo
Xem bảng trạng thái toàn bộ pipeline và báo cáo đối chuẩn Table 3.

In [ ]:
# ── 5.1. Xem Dashboard Trạng Thái Pipeline ─────────────────
!python run_pipeline.py status --output_dir outputs/pcb_synthesis

# ── 5.2. Hiển thị Báo Cáo Đối Chuẩn (Benchmark Report) ─────
from IPython.display import Markdown, display
report_file = Path("outputs/pcb_synthesis/benchmark/benchmark_report.md")
if report_file.exists():
    display(Markdown(report_file.read_text(encoding="utf-8")))
else:
    print("Chưa có file benchmark_report.md. Hãy hoàn tất Step 4 để xem bảng so sánh Table 3!")

In [ ]:
# ── 5.3. Hiển thị mẫu ảnh khuyết tật tổng hợp đã qua lọc ──
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

accepted_imgs = list(Path("outputs/pcb_synthesis/filtered/accepted/images").glob("*.jpg"))
if not accepted_imgs:
    accepted_imgs = list(Path("outputs/pcb_synthesis/generated").glob("*.jpg"))

if accepted_imgs:
    samples = accepted_imgs[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_p in zip(axes.flat, samples):
        img = cv2.imread(str(img_p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(img_p.name, fontsize=10)
        ax.axis("off")
    plt.suptitle("Sample Synthetic Defect Images Generated by Inpainting Diffusion", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Chưa tìm thấy ảnh tổng hợp nào để hiển thị.")